## **Notebook Configuration**

In [0]:
# -------------------------------------------
# Day 2 - Silver Layer Feature Engineering
# -------------------------------------------

BRONZE_TABLE = "workspace.ecommerce.bronze_events"
SILVER_TABLE = "workspace.ecommerce.silver_user_features"

In [0]:
# Load Bronze Data
bronze_df = spark.table(BRONZE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(5))

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-11T17:14:26.000Z,view,1004739,2053013555631882655,electronics.smartphone,xiaomi,188.02,507682847,dfc44355-6b7b-4bcd-8feb-89c022ed254d
2019-10-11T17:14:27.000Z,view,26300182,2053013563424899933,null,trollbeads,194.08,541632398,ca1496c2-e2af-4a38-bb85-adf72e52d31f
2019-10-11T17:14:27.000Z,view,1005003,2053013555631882655,electronics.smartphone,huawei,252.23,514447368,dec3ce89-b93e-4379-9d2d-7521e4304f3d
2019-10-11T17:14:27.000Z,view,22400122,2053013554474254687,electronics.audio.microphone,trust,110.94,519285494,81c776b3-fd06-449e-a982-108be660da2b
2019-10-11T17:14:27.000Z,view,15100227,2053013557024391671,null,null,731.04,549462175,d244b901-5a38-41f3-a184-719eeb972d6a


In [0]:
# Basic Data Validation,ensures the dataset structure hasn't changed.
print("Total Events:", bronze_df.count())

spark.sql("""
SELECT event_type, COUNT(*)
FROM workspace.ecommerce.bronze_events
GROUP BY event_type
""").show()

Total Events: 109950743
+----------+---------+
|event_type| COUNT(*)|
+----------+---------+
|  purchase|  1659788|
|      cart|  3955446|
|      view|104335509|
+----------+---------+



## **Feature Engineering Logic**

In [0]:
# Aggregate user behaviour, these converts 110m event-level data --> user-level fearures

from pyspark.sql import functions as F

user_features_df = bronze_df.groupBy("user_id").agg(
  F.count("*").alias("total_events"),

  F.count(
       F.when(F.col("event_type") == "purchase", True)
  ).alias("total_purchases"),

  F.sum("price").alias("total_spent"),
  
  F.avg("price").alias("avg_price")
)

In [0]:
# Add Behavioral Ratios, common in ML systems, this tell the model how often a user converts

user_features_df = user_features_df.withColumn(
    "purchase_rate",
    F.col("total_purchases") / F.col("total_events")
)

In [0]:
# Handle NULL Values, as some users never purchased anything so total_spent, avg_price may be null so need to replace them safely . This avoids model errors later

user_features_df = user_features_df.fillna({
    "total_purchases": 0.0,
    "total_spent": 0.0,
    "avg_price": 0.0,
  
})


In [0]:
# Add Metadata Column,Now your table records when features were generated

user_features_df = user_features_df.withColumn(
    "feature_generation_time",
    F.current_timestamp()
)

## **Validate Feature Table**

In [0]:
# Before Writing Silver data , always insperct

# Check Distribution:
user_features_df.describe().show()
# Check Record Count:
print("Total users:",user_features_df.count())
#Check for duplicates: This confirms one row per user,ensure unique users
user_features_df.select("user_id").distinct().count()

+-------+------------------+-----------------+------------------+------------------+-----------------+--------------------+
|summary|           user_id|     total_events|   total_purchases|       total_spent|        avg_price|       purchase_rate|
+-------+------------------+-----------------+------------------+------------------+-----------------+--------------------+
|  count|           5316649|          5316649|           5316649|           5316649|          5316649|             5316649|
|   mean|5.47298119290999E8|20.68045925168278| 0.312186868081756| 6031.141646012436|310.1861120623595|0.011712526364925108|
| stddev|2.26738817378748E7|53.91671605953241|1.7878229513587522|17755.080208659638|315.1068799114791|0.044082393977567746|
|    min|          10300217|                1|                 0|               0.0|              0.0|                 0.0|
|    max|         579969851|            22929|               640| 5187451.929999998|          2574.07|                 1.0|
+-------

5316649

## **Now Store Feature SILVER TABLE**

In [0]:
# Save "workspace.ecommerce.silver_users_features" table as SILVER_TABLE (user behavioral feature table) , as we linked it very before at 1st line

user_features_df.write.format("delta") \
    .mode("overwrite") \
        .saveAsTable(SILVER_TABLE)

## **Verify Silver Table**

In [0]:
# First confirm it exists
spark.sql("SHOW TABLES IN workspace.ecommerce")

DataFrame[database: string, tableName: string, isTemporary: boolean]

In [0]:
#Then Inspect:
spark.table(SILVER_TABLE).show(10)


+---------+------------+---------+------------------+------------------+
|  user_id|total_events|purchases|       total_spent|         avg_price|
+---------+------------+---------+------------------+------------------+
|571787593|          86|        0| 22155.96000000001|257.62744186046524|
|513881389|         226|       13|          66781.49|295.49331858407083|
|515137025|          53|        0|1341.7999999999993| 25.31698113207546|
|561763782|         108|        0|25270.529999999995|233.98638888888885|
|517056737|          10|        0|           2430.86|           243.086|
|571864123|          57|        1|          11213.09|196.72087719298247|
|537275608|         128|        0|          19490.14|      152.26671875|
|518985166|         168|        3|           24551.1|          146.1375|
|513512264|          14|        0|           8000.32| 571.4514285714286|
|572147643|           7|        0|           1170.74|167.24857142857144|
+---------+------------+---------+-----------------

In [0]:
# Quick Statistical Check
spark.table("workspace.ecommerce.silver_user_features").describe().show()

+-------+-------------------+-----------------+------------------+------------------+-----------------+
|summary|            user_id|     total_events|         purchases|       total_spent|        avg_price|
+-------+-------------------+-----------------+------------------+------------------+-----------------+
|  count|            5316649|          5316649|           5316649|           5316649|          5316649|
|   mean| 5.47298119290999E8|20.68045925168278| 0.312186868081756| 6031.141646012252|310.1861120622935|
| stddev|2.267388173787481E7|53.91671605953256|1.7878229513587518|17755.080208659634|315.1068799114791|
|    min|           10300217|                1|                 0|               0.0|              0.0|
|    max|          579969851|            22929|               640| 5187451.929999998|          2574.07|
+-------+-------------------+-----------------+------------------+------------------+-----------------+



In [0]:
# Check How Many users exists
spark.sql("""
SELECT COUNT(*) FROM workspace.ecommerce.silver_user_features""").show()

+--------+
|COUNT(*)|
+--------+
| 5316649|
+--------+



## **Silver Table Optimization(Optional)**

In [0]:
#Silver tables are queried more often than Bronze. So we can optionally optimize.Not mandatory, but useful.
spark.sql("""
OPTIMIZE workspace.ecommerce.silver_user_features ZORDER BY (user_id)""")


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,